# GroupDNA — WhatsApp Group Chat Analyzer
**Project:** GroupDNA (Week 1 Minor Project — The Unlox Academy)
**Name:** Sirajudeen
**Roll Number:** UNXBSARCIST-1360
**Batch:** DATA SCIENCE
**Date:** 22-8-2026

Built using only Python fundamentals + NumPy. No pandas, no matplotlib, no regex, no collections.


## Setup

Upload `hostel_bois.txt` to this Colab session (left sidebar → folder icon → upload), or place it next to this notebook if running locally. Then run the cell below.


In [10]:
import numpy as np
from datetime import datetime

FILE_PATH = 'hostel_bois.txt'   # adjust if your path differs (e.g. '/content/hostel_bois.txt')


## Feature 1: The Chat Parser

Reads `hostel_bois.txt` line by line and extracts `(timestamp, sender, text)` for every real message.
Handles system messages, media-omitted, deleted messages, multi-line continuations, and empty lines.


In [11]:
def parse_chat(filepath):
    """Parse a WhatsApp export into a list of message dicts.
    Returns: messages, system_count, media_count (per sender), deleted_count (per sender)
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.read().split('\n')

    messages = []
    system_count = 0
    media_count = {}
    deleted_count = {}
    last_message = None  # for multi-line continuations

    for line in lines:
        if line.strip() == '':
            continue  # (e) skip empty lines silently

        # (d) multi-line message: doesn't start with a DD/MM/YY date pattern
        first8 = line[:8]
        looks_like_date = (
            len(first8) == 8 and first8[2] == '/' and first8[5] == '/'
            and first8[0:2].isdigit() and first8[3:5].isdigit() and first8[6:8].isdigit()
        )
        if not looks_like_date:
            if last_message is not None:
                last_message['text'] += ' ' + line.strip()
            continue

        parts = line.split(' - ', 1)
        if len(parts) != 2:
            system_count += 1
            last_message = None
            continue

        timestamp, rest = parts

        # (a) system message: no ": " -> no real sender/message split possible
        if ': ' not in rest:
            system_count += 1
            last_message = None
            continue

        sender, text = rest.split(': ', 1)
        msg = {'timestamp': timestamp, 'sender': sender, 'text': text,
               'is_media': False, 'is_deleted': False}

        # (b) media omitted
        if text.strip() == '<Media omitted>':
            msg['is_media'] = True
            media_count[sender] = media_count.get(sender, 0) + 1
        # (c) deleted message
        elif text.strip() == 'This message was deleted':
            msg['is_deleted'] = True
            deleted_count[sender] = deleted_count.get(sender, 0) + 1

        messages.append(msg)
        last_message = msg

    return messages, system_count, media_count, deleted_count


messages, system_count, media_count, deleted_count = parse_chat(FILE_PATH)
participants = sorted(set(m['sender'] for m in messages))
total_media = sum(media_count.values())
total_deleted = sum(deleted_count.values())

print(f"Successfully parsed {len(messages)} messages from {len(participants)} participants, "
      f"skipped {system_count} system messages, {total_media} media-omitted, "
      f"{total_deleted} deleted messages.")


Successfully parsed 3174 messages from 6 participants, skipped 4 system messages, 32 media-omitted, 15 deleted messages.


## Feature 2: Group Overview

Headline stats: total messages, date range, participant count, and per-person message share.


In [12]:
def get_date(ts):
    return ts.split(',')[0]

per_person_count = {}
for m in messages:
    per_person_count[m['sender']] = per_person_count.get(m['sender'], 0) + 1

all_dates = [get_date(m['timestamp']) for m in messages]
d1 = datetime.strptime(all_dates[0], '%d/%m/%y')
d2 = datetime.strptime(all_dates[-1], '%d/%m/%y')
total_days = (d2 - d1).days + 1
total_messages = len(messages)
sorted_people = sorted(per_person_count.items(), key=lambda x: x[1], reverse=True)

print("=" * 60)
print(" GROUP OVERVIEW")
print("=" * 60)
print(f" Period       : {d1.strftime('%d %B %Y')} to {d2.strftime('%d %B %Y')} ({total_days} days)")
print(f" Total msgs   : {total_messages}")
print(f" Participants : {len(participants)}")
print(" MESSAGES PER PERSON")
for person, count in sorted_people:
    pct = count / total_messages * 100
    print(f"  {person:<10}: {count:>5} ({pct:>4.1f}%)")


 GROUP OVERVIEW
 Period       : 01 April 2024 to 30 May 2024 (60 days)
 Total msgs   : 3174
 Participants : 6
 MESSAGES PER PERSON
  Rahul     :   953 (30.0%)
  Priya     :   718 (22.6%)
  Neha      :   635 (20.0%)
  Aman      :   490 (15.4%)
  Karan     :   354 (11.2%)
  Vikas     :    24 ( 0.8%)


## Feature 3: Most Active Day and Hour

Finds the single busiest day, and the hour of day (summed across all 60 days) with the most messages.


In [13]:
day_counts = {}
hour_counts = {}
for m in messages:
    date = get_date(m['timestamp'])
    hour = int(m['timestamp'].split(', ')[1].split(':')[0])
    day_counts[date] = day_counts.get(date, 0) + 1
    hour_counts[hour] = hour_counts.get(hour, 0) + 1

busiest_day = max(day_counts.items(), key=lambda x: x[1])
busiest_hour = max(hour_counts.items(), key=lambda x: x[1])
bd_dt = datetime.strptime(busiest_day[0], '%d/%m/%y')

print(f" Busiest day  : {bd_dt.strftime('%d %B %Y')} ({busiest_day[1]} messages)")
print(f" Busiest hour : {busiest_hour[0]:02d}:00 - {(busiest_hour[0]+1)%24:02d}:00 "
      f"(avg {busiest_hour[1]/total_days:.1f} messages per day)")


 Busiest day  : 04 May 2024 (76 messages)
 Busiest hour : 18:00 - 19:00 (avg 4.1 messages per day)


## Feature 4: Activity Heatmap (NumPy)

Builds a 6×24 NumPy matrix — rows = participants, columns = hours of day — then renders it as a
block-character heatmap, shaded relative to each person's own peak hour.


In [14]:
person_index = {p: i for i, p in enumerate(participants)}
heatmap = np.zeros((len(participants), 24), dtype=int)

for m in messages:
    p_idx = person_index[m['sender']]
    hour = int(m['timestamp'].split(', ')[1].split(':')[0])
    heatmap[p_idx, hour] += 1

def shade(ratio):
    if ratio <= 0.25:
        return '.  '
    elif ratio <= 0.5:
        return '░  '
    elif ratio <= 0.75:
        return '▒  '
    else:
        return '█  '

def render_heatmap(matrix, people):
    print(" ACTIVITY HEATMAP (messages by hour, columns every 3 hours)")
    header = "        " + "".join(f"{h:02d}   " for h in range(0, 24, 3))
    print(header)
    for i, person in enumerate(people):
        row = matrix[i]
        row_max = row.max() if row.max() > 0 else 1
        line = f"  {person:<7}"
        for h in range(0, 24, 3):
            line += shade(row[h] / row_max)
        print(line)

render_heatmap(heatmap, participants)

# sanity: total messages per person from the heatmap should match per_person_count
for i, p in enumerate(participants):
    assert heatmap[i].sum() == per_person_count[p], f"mismatch for {p}"
print("\nHeatmap row totals verified against per-person message counts.")


 ACTIVITY HEATMAP (messages by hour, columns every 3 hours)
        00   03   06   09   12   15   18   21   
  Aman   ▒  ▒  .  .  .  .  .  .  
  Karan  .  .  .  ░  █  ▒  ▒  ░  
  Neha   .  .  .  █  ▒  .  █  ░  
  Priya  .  .  .  █  █  ░  ▒  ░  
  Rahul  .  .  .  .  ▒  ▒  █  █  
  Vikas  .  .  .  ░  ▒  ░  ▒  ░  

Heatmap row totals verified against per-person message counts.


## Feature 5: Top Words

Tokenizes every real message (skipping media/deleted), lowercases, strips punctuation, removes
stop words, and prints the top 10 group-wide words with proportional bars.


In [15]:
STOP_WORDS = {
    'i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for', 'you',
    'it', 'this', 'that', 'my', 'me', 'we', 'are', 'be', 'at', 'so', 'do',
    'not', 'if', 'was', 'with', 'how', 'about', 'am', 'today', 'he', 'his',
    'have', 'just', 'which', 'everyone', 'telling', 'from', 'up', 'one',
    'had', 'started', 'they', 'her', 'she', 'him', 'them', 'their', 'there',
    'here', 'what', 'when', 'where', 'who', 'why', 'can', 'will', 'would',
    'could', 'should', 'been', 'being', 'then', 'than', 'but', 'as', 'all',
    'some', 'any', 'no', 'yes', 'ok', 'out', 'into', 'over', 'after',
    'before', 'still', 'also', 'very', 'really', 'too', 'only', 'even',
    'got', 'get', 'going', 'went', 'go', 'did', 'does', 'were', 'us', 'our',
    'your', 'yours', 'im', 'its', 'dont', 'doesnt', 'didnt', 'cant',
    'wasnt', 'isnt', 'more', 'most', 'much', 'many', 'something',
    'anything', 'nothing', 'everything', 'someone', 'anyone', 'because',
    'while', 'during', 'until', 'since', 'though',
}
PUNCT = '.,!?"\'()[]{}:;-_/\\<>@#$%^&*+=~`'

def clean_word(w):
    return w.strip(PUNCT).lower()

word_counts = {}
real_messages = [m for m in messages if not m['is_media'] and not m['is_deleted']]

for m in real_messages:
    for raw_word in m['text'].split():
        w = clean_word(raw_word)
        if w == '' or w in STOP_WORDS or not w.isalpha() or len(w) < 2:
            continue
        word_counts[w] = word_counts.get(w, 0) + 1

top_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:10]

print(" THIS GROUP'S FAVOURITE WORDS")
max_count = top_words[0][1] if top_words else 1
for word, count in top_words:
    bar_len = int((count / max_count) * 20)
    print(f"  {word:<10} {chr(0x2588) * bar_len} {count}")


 THIS GROUP'S FAVOURITE WORDS
  guys       ████████████████████ 318
  hai        ████████████████ 268
  bhai       ██████████ 160
  scene      █████████ 145
  entire     █████████ 145
  please     ████████ 141
  yaar       ████████ 139
  kya        ████████ 133
  now        ███████ 121
  came       ███████ 116


## Feature 6: Response Speed & Silent Streaks

(a) Average response gap — time between someone else's message and this person's next reply.
(b) Longest run of consecutive days with zero messages, per person.


In [16]:
parsed_messages = [(datetime.strptime(m['timestamp'], '%d/%m/%y, %H:%M'), m['sender']) for m in messages]

response_gaps = {p: [] for p in participants}
for i in range(1, len(parsed_messages)):
    prev_dt, prev_sender = parsed_messages[i - 1]
    curr_dt, curr_sender = parsed_messages[i]
    if curr_sender != prev_sender:
        gap_seconds = (curr_dt - prev_dt).total_seconds()
        response_gaps[curr_sender].append(gap_seconds)

avg_response = {p: (sum(g) / len(g) if g else 0) for p, g in response_gaps.items()}

def format_duration(seconds):
    if seconds < 3600:
        return f"{seconds/60:.1f} minutes"
    return f"{seconds/3600:.1f} hours"

fastest = min(avg_response.items(), key=lambda x: x[1])
slowest = max(avg_response.items(), key=lambda x: x[1])

print(" RESPONSE PATTERNS")
print(f"  Fastest replier : {fastest[0]} (avg {format_duration(fastest[1])})")
print(f"  Slowest replier : {slowest[0]} (avg {format_duration(slowest[1])})")

# --- Silent streaks ---
active_days_per_person = {p: set() for p in participants}
for m in messages:
    active_days_per_person[m['sender']].add(get_date(m['timestamp']))

all_days = []
d = d1
while d <= d2:
    all_days.append(d.strftime('%d/%m/%y'))
    d = datetime.fromordinal(d.toordinal() + 1)

silent_streaks, silent_ranges = {}, {}
for p in participants:
    active = active_days_per_person[p]
    longest, current, streak_start = 0, 0, None
    best_start, best_end = None, None
    for day in all_days:
        if day not in active:
            if current == 0:
                streak_start = day
            current += 1
            if current > longest:
                longest = current
                best_start, best_end = streak_start, day
        else:
            current = 0
    silent_streaks[p] = longest
    silent_ranges[p] = (best_start, best_end)

print(" LONGEST SILENT STREAKS (consecutive days with zero messages)")
for p, streak in sorted(silent_streaks.items(), key=lambda x: x[1], reverse=True):
    if streak == 0:
        print(f"  {p:<10}: 0 days (never went silent)")
    else:
        s, e = silent_ranges[p]
        print(f"  {p:<10}: {streak} days ({s} to {e})")


 RESPONSE PATTERNS
  Fastest replier : Rahul (avg 34.9 minutes)
  Slowest replier : Aman (avg 55.4 minutes)
 LONGEST SILENT STREAKS (consecutive days with zero messages)
  Vikas     : 11 days (23/04/24 to 03/05/24)
  Aman      : 0 days (never went silent)
  Karan     : 0 days (never went silent)
  Neha      : 0 days (never went silent)
  Priya     : 0 days (never went silent)
  Rahul     : 0 days (never went silent)


## Feature 7: Personality Archetype Detection

One scoring function per archetype (Section 7 of the brief). Each person is assigned exactly one
archetype: threshold archetypes (Spammer, Night Owl, Storyteller, Drama Queen, Ghost) are awarded to
the highest-scoring qualifying, not-yet-claimed person; Group Mom goes to the highest caring-keyword
scorer among who's left; anyone still unassigned falls back to whichever of Comedian / Question Master
they score higher on.

**Ninth archetype (bonus, invented):** `THE LATE-NIGHT PHILOSOPHER` — uses reflective words like
"life", "time", "meaning", "wonder" in messages sent after midnight. Specific to the late-night hostel
"gyaan session" trope common in Indian college life.


In [17]:
CARING_KEYWORDS = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you',
                    'please', 'reminder', 'drink water', "don't forget"]
LAUGH_WORDS = ['lol', 'lmao', 'haha', 'rofl', 'lmfao']
PHILOSOPHER_WORDS = ['life', 'time', 'meaning', 'wonder', 'exist', 'universe']

def score_spammer(msgs_all, person):
    # avg length of consecutive message bursts by this person (no one else speaking in between)
    bursts, current_burst = [], 0
    for m in msgs_all:
        if m['sender'] == person:
            current_burst += 1
        else:
            if current_burst > 0:
                bursts.append(current_burst)
            current_burst = 0
    if current_burst > 0:
        bursts.append(current_burst)
    return sum(bursts) / len(bursts) if bursts else 0

def score_group_mom(person_msgs):
    score = 0
    for m in person_msgs:
        text_lower = m['text'].lower()
        for kw in CARING_KEYWORDS:
            score += text_lower.count(kw)
    return score

def score_night_owl(person_msgs):
    if not person_msgs:
        return 0
    night_count = sum(1 for m in person_msgs
                       if int(m['timestamp'].split(', ')[1].split(':')[0]) >= 23
                       or int(m['timestamp'].split(', ')[1].split(':')[0]) <= 4)
    return night_count / len(person_msgs) * 100

def score_storyteller(person_msgs):
    real = [m for m in person_msgs if not m['is_media'] and not m['is_deleted']]
    if not real:
        return 0
    return sum(len(m['text'].split()) for m in real) / len(real)

def score_drama_queen(person_msgs):
    real = [m for m in person_msgs if not m['is_media'] and not m['is_deleted']]
    if not real:
        return 0
    drama_count = 0
    for m in real:
        text = m['text']
        alpha_text = ''.join(c for c in text if c.isalpha())
        is_caps = len(alpha_text) >= 3 and alpha_text.isupper()
        has_excl = text.count('!') >= 2
        if is_caps or has_excl:
            drama_count += 1
    return drama_count / len(real) * 100

def score_ghost(person, active_days, total_days):
    silent_days = total_days - len(active_days[person])
    return silent_days / total_days * 100

def score_comedian(person_msgs):
    real = [m for m in person_msgs if not m['is_media'] and not m['is_deleted']]
    if not real:
        return 0
    laugh_count = sum(1 for m in real if any(lw in m['text'].lower() for lw in LAUGH_WORDS))
    return laugh_count / len(real) * 100

def score_question_master(person_msgs):
    real = [m for m in person_msgs if not m['is_media'] and not m['is_deleted']]
    if not real:
        return 0
    q_count = sum(1 for m in real if m['text'].strip().endswith('?'))
    return q_count / len(real) * 100

def score_philosopher(person_msgs):
    real = [m for m in person_msgs if not m['is_media'] and not m['is_deleted']]
    night = [m for m in real if int(m['timestamp'].split(', ')[1].split(':')[0]) >= 23
             or int(m['timestamp'].split(', ')[1].split(':')[0]) <= 4]
    if not night:
        return 0
    hits = sum(1 for m in night if any(w in m['text'].lower() for w in PHILOSOPHER_WORDS))
    return hits / len(night) * 100

msgs_by_person = {p: [m for m in messages if m['sender'] == p] for p in participants}

archetype_scores = {}
for p in participants:
    pm = msgs_by_person[p]
    archetype_scores[p] = {
        'THE SPAMMER': score_spammer(messages, p),
        'THE GROUP MOM': score_group_mom(pm),
        'THE NIGHT OWL': score_night_owl(pm),
        'THE STORYTELLER': score_storyteller(pm),
        'THE DRAMA QUEEN': score_drama_queen(pm),
        'THE GHOST': score_ghost(p, active_days_per_person, total_days),
        'THE COMEDIAN': score_comedian(pm),
        'THE QUESTION MASTER': score_question_master(pm),
        'THE LATE-NIGHT PHILOSOPHER': score_philosopher(pm),
    }

# Threshold rules from Section 7 of the brief
THRESHOLDS = {
    'THE SPAMMER': 3, 'THE NIGHT OWL': 60, 'THE STORYTELLER': 30,
    'THE DRAMA QUEEN': 30, 'THE GHOST': 60,
}
threshold_archetypes = ['THE SPAMMER', 'THE NIGHT OWL', 'THE STORYTELLER',
                         'THE DRAMA QUEEN', 'THE GHOST']

assigned = {}
# Tie-break rule: for each threshold archetype, whoever scores HIGHEST among
# qualifying, not-yet-assigned people gets it (documented per brief's Section 7 note).
for arch in threshold_archetypes:
    candidates = [(p, archetype_scores[p][arch]) for p in participants
                  if p not in assigned and archetype_scores[p][arch] >= THRESHOLDS[arch]]
    if candidates:
        winner = max(candidates, key=lambda x: x[1])
        assigned[winner[0]] = arch

remaining = [p for p in participants if p not in assigned]
if remaining:
    mom_candidate = max(remaining, key=lambda p: archetype_scores[p]['THE GROUP MOM'])
    if archetype_scores[mom_candidate]['THE GROUP MOM'] > 0:
        assigned[mom_candidate] = 'THE GROUP MOM'

remaining = [p for p in participants if p not in assigned]
for p in remaining:
    scores = archetype_scores[p]
    fallback = max(['THE COMEDIAN', 'THE QUESTION MASTER'], key=lambda a: scores[a])
    assigned[p] = fallback

print(" PERSONALITY ARCHETYPES")
for p in participants:
    arch = assigned[p]
    score = archetype_scores[p][arch]
    print(f"  {p:<10} -> {arch}  (score: {score:.1f})")


 PERSONALITY ARCHETYPES
  Aman       -> THE NIGHT OWL  (score: 79.8)
  Karan      -> THE STORYTELLER  (score: 57.0)
  Neha       -> THE DRAMA QUEEN  (score: 63.3)
  Priya      -> THE GROUP MOM  (score: 621.0)
  Rahul      -> THE SPAMMER  (score: 4.5)
  Vikas      -> THE GHOST  (score: 73.3)


## Feature 8: The Final Report

Everything above, wrapped into one clean, screenshot-worthy printed report.


In [18]:
def print_final_report():
    W = 62
    print("=" * W)
    print(' GROUPDNA REPORT — "Hostel Bois 4ever"'.center(W))
    print(f" {total_days} days • {total_messages:,} messages • {len(participants)} members".center(W))
    print("=" * W)
    print(f" Period       : {d1.strftime('%d %B %Y')} to {d2.strftime('%d %B %Y')}")
    print(f" Busiest day  : {bd_dt.strftime('%d %B %Y')} ({busiest_day[1]} messages)")
    print(f" Busiest hour : {busiest_hour[0]:02d}:00 - {(busiest_hour[0]+1)%24:02d}:00")
    print()
    print(" MESSAGES PER PERSON")
    max_p = sorted_people[0][1]
    for person, count in sorted_people:
        pct = count / total_messages * 100
        bar_len = int((count / max_p) * 20)
        print(f"  {person:<7}{chr(0x2588)*bar_len:<21} {count:>5} ({pct:>4.1f}%)")
    print()
    render_heatmap(heatmap, participants)
    print()
    print(" THIS GROUP'S FAVOURITE WORDS")
    for word, count in top_words:
        bar_len = int((count / max_count) * 20)
        print(f"  {word:<10} {chr(0x2588)*bar_len} {count}")
    print()
    print(" RESPONSE PATTERNS")
    print(f"  Fastest replier : {fastest[0]} (avg {format_duration(fastest[1])})")
    print(f"  Slowest replier : {slowest[0]} (avg {format_duration(slowest[1])})")
    print()
    print(" LONGEST SILENT STREAKS")
    for p, streak in sorted(silent_streaks.items(), key=lambda x: x[1], reverse=True):
        if streak == 0:
            print(f"  {p:<7}: 0 days")
        else:
            print(f"  {p:<7}: {streak} days")
    print()
    print(" PERSONALITY ARCHETYPES")
    for p in participants:
        arch = assigned[p]
        print(f"  {p:<7} -> {arch}")
    print()
    print("=" * W)
    print(" Generated by GroupDNA • Built with Python + NumPy".center(W))
    print("=" * W)

print_final_report()


             GROUPDNA REPORT — "Hostel Bois 4ever"            
             60 days • 3,174 messages • 6 members             
 Period       : 01 April 2024 to 30 May 2024
 Busiest day  : 04 May 2024 (76 messages)
 Busiest hour : 18:00 - 19:00

 MESSAGES PER PERSON
  Rahul  ████████████████████    953 (30.0%)
  Priya  ███████████████         718 (22.6%)
  Neha   █████████████           635 (20.0%)
  Aman   ██████████              490 (15.4%)
  Karan  ███████                 354 (11.2%)
  Vikas                           24 ( 0.8%)

 ACTIVITY HEATMAP (messages by hour, columns every 3 hours)
        00   03   06   09   12   15   18   21   
  Aman   ▒  ▒  .  .  .  .  .  .  
  Karan  .  .  .  ░  █  ▒  ▒  ░  
  Neha   .  .  .  █  ▒  .  █  ░  
  Priya  .  .  .  █  █  ░  ▒  ░  
  Rahul  .  .  .  .  ▒  ▒  █  █  
  Vikas  .  .  .  ░  ▒  ░  ▒  ░  

 THIS GROUP'S FAVOURITE WORDS
  guys       ████████████████████ 318
  hai        ████████████████ 268
  bhai       ██████████ 160
  scene      ███████

## Bonus: Running This on Your Own Chat

1. Open WhatsApp on your phone → open the group chat → **More options → More → Export chat → Without media**.
2. This produces a `.txt` file. AirDrop / email / save it to your Drive.
3. Upload it to this Colab session (don't upload it to GitHub — keep it private).
4. Change `FILE_PATH` at the top of this notebook to your file's name, and re-run all cells.
5. Screenshot the final report from `print_final_report()` and use it for your LinkedIn post — never share the raw chat file publicly.


## Reflection

-**Hardest part:** Getting the chat parser (Feature 1) to correctly handle every edge case — especially telling system messages apart from real ones, since both have a timestamp and a dash but only real messages have a sender followed by a colon. Debugging the archetype tie-breaking logic in Feature 7 was also tricky, since multiple people could qualify for the same archetype and I had to decide who "wins" it.

**What I'd do differently:** I'd build a small set of test cases for the parser first (a few lines of each edge case — system message, media, deleted, multi-line) before running it on the full 3,174-line file, instead of debugging against the whole dataset at once.

**My own chat's archetype (optional):** Not run yet — planning to try this on my own hostel group chat as the bonus and see what comes out.

**AI assistance disclosure:** Claude was used to help design the parsing logic, the archetype scoring approach, and the final report formatting. All code was reviewed, adapted, and restructured in my own style before submission.